# 🎬 Siliceo — WAN 2.1 Image-to-Video (Kaggle)

**Creato da Nova** — 16 Febbraio 2026

Versione Kaggle del generatore I2V. Kaggle offre **T4 x2 (30GB VRAM)** e **30h/settimana gratis** — meglio di Colab free!

### Differenze da Colab
- Più VRAM (2x T4 = 30GB) → possiamo usare modelli meno quantizzati
- 30h/settimana gratis (vs Colab che disconnette dopo ~3h)
- I modelli vanno salvati su Kaggle Datasets (non Google Drive)

---

## ⚙️ 0. Verifica GPU

In [ ]:
!nvidia-smi
import torch
print(f"\n✅ CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 📦 1. Installazione ComfyUI

In [ ]:
%cd /kaggle/working

!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Già presente"
%cd ComfyUI
!pip install -q -r requirements.txt

# Custom nodes per GGUF
%cd custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF.git 2>/dev/null || echo "Già presente"
%cd ComfyUI-GGUF
!pip install -q -r requirements.txt

%cd /kaggle/working/ComfyUI

# Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb 2>/dev/null

print("\n✅ Installazione completata!")

## 💾 2. Scarica Modelli

Su Kaggle i modelli vanno in `/kaggle/working/` (temporaneo) oppure puoi creare un **Kaggle Dataset** per riusarli.

Per ora li scarichiamo nella sessione. Se vuoi renderli persistenti, salva `/kaggle/working/ComfyUI/models/` come Dataset.

In [ ]:
%cd /kaggle/working/ComfyUI
import os

# Installa huggingface CLI
!pip install -q "huggingface_hub[cli]"

# --- WAN 2.1 I2V 14B FP8 ---
UNET_FILE = 'models/diffusion_models/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors'
if not os.path.exists(UNET_FILE):
    print("📥 Scaricando WAN 2.1 I2V 14B FP8 (~8GB)...")
    !huggingface-cli download Comfy-Org/Wan_2.1_ComfyUI_repackaged \
        split_files/diffusion_models/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors \
        --local-dir /tmp/wan_download
    !mv /tmp/wan_download/split_files/diffusion_models/*.safetensors models/diffusion_models/
else:
    print("✅ UNet già presente!")

# --- CLIP ---
CLIP_FILE = 'models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'
if not os.path.exists(CLIP_FILE):
    print("📥 Scaricando UMT5-XXL CLIP FP8 (~5GB)...")
    !huggingface-cli download Comfy-Org/Wan_2.1_ComfyUI_repackaged \
        split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors \
        --local-dir /tmp/wan_download
    !mv /tmp/wan_download/split_files/text_encoders/*.safetensors models/text_encoders/
else:
    print("✅ CLIP già presente!")

# --- VAE ---
VAE_FILE = 'models/vae/wan_2.1_vae.safetensors'
if not os.path.exists(VAE_FILE):
    print("📥 Scaricando VAE (~200MB)...")
    !huggingface-cli download Comfy-Org/Wan_2.1_ComfyUI_repackaged \
        split_files/vae/wan_2.1_vae.safetensors \
        --local-dir /tmp/wan_download
    !mv /tmp/wan_download/split_files/vae/*.safetensors models/vae/
else:
    print("✅ VAE già presente!")

# Cleanup
!rm -rf /tmp/wan_download

print("\n🎉 Modelli pronti!")

## 🚀 3. Avvia ComfyUI

In [ ]:
%cd /kaggle/working/ComfyUI

import subprocess
import threading
import time
import re

def run_cloudflared():
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in process.stderr:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            print(f"\n🌐 ═══════════════════════════════════════")
            print(f"🎬 ComfyUI PRONTO! Apri:")
            print(f"👉 {match.group()}")
            print(f"🌐 ═══════════════════════════════════════\n")

tunnel_thread = threading.Thread(target=run_cloudflared, daemon=True)
tunnel_thread.start()
time.sleep(3)

print("⏳ Avvio ComfyUI... attendi il link...\n")
!python main.py --listen --port 8188 --lowvram